In [18]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional, Set

import numpy as np
import pandas as pd
from pymongo import MongoClient
import yaml


In [19]:
def load_config(path: str | Path = "../config/config.yaml") -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        path = Path("config/config.yaml")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


config = load_config()
mongo_config = config["mongo"]

In [23]:
client = MongoClient(mongo_config.get('url'))
db = client[mongo_config.get('db')]
col_schedule = db[mongo_config.get('collection').get('collection_schedule')]
df_schedule = pd.DataFrame(col_schedule.find({'season': config.get("season").get("year")}))
check_games = df_schedule['game_id'].sample(5, random_state=1909).values.tolist()

In [25]:
col_team_stats = db[mongo_config.get('collection').get('collection_team_game_stats')]
col_player_stats = db[mongo_config.get('collection').get('collection_player_game_stats')]

df_team_stats = pd.DataFrame(col_team_stats.find({'game_id': {"$in": check_games}}))
df_player_stast = pd.DataFrame(col_player_stats.find({'game_id': {"$in": check_games}}))

In [26]:
pd.set_option("display.max_rows", None)

for game in check_games:
    team_stats = df_team_stats[df_team_stats.game_id == game]
    team_stats = team_stats.sort_values(by='game_venue', ascending=False).reset_index(drop=True)
    game_info = team_stats[['game_id', 'game_date', 'week', 'team_name', 'game_venue']]
    stats = pd.json_normalize(team_stats.stats)
    stats_df = stats[[col for col in stats.columns if 'ft.' in col]]
    
    final_game = pd.merge(
        game_info,
        stats_df,
        left_index=True,
        right_index=True
    )
    display(final_game.T)
    print("\n" + "-" * 80 + "\n")


,0,1
game_id,1910804,1910804
game_date,2026-01-14 17:30:00,2026-01-14 17:30:00
week,17,17
team_name,Wolfsburg,St. Pauli
game_venue,home,away
ft.possession,0.608616,0.391384
ft.goals,2,1
ft.shots,13,9
ft.shot_goal,2,1
ft.shot_on_target,5,4



--------------------------------------------------------------------------------



,0,1
game_id,1910777,1910777
game_date,2026-03-14 14:30:00,2026-03-14 14:30:00
week,26,26
team_name,Borussia Dortmund,Augsburg
game_venue,home,away
ft.possession,0.599613,0.400387
ft.goals,2,0
ft.shots,16,7
ft.shot_goal,2,0
ft.shot_on_target,5,3



--------------------------------------------------------------------------------



,0,1
game_id,1910835,1910835
game_date,2026-02-07 14:30:00,2026-02-07 14:30:00
week,21,21
team_name,Freiburg,Werder Bremen
game_venue,home,away
ft.possession,0.334728,0.665272
ft.goals,1,0
ft.shots,7,17
ft.shot_goal,1,0
ft.shot_on_target,2,5



--------------------------------------------------------------------------------



,0,1
game_id,1910874,1910874
game_date,2026-04-26 14:30:00,2026-04-26 14:30:00
week,31,31
team_name,VfB Stuttgart,Werder Bremen
game_venue,home,away
ft.possession,0.682156,0.317844
ft.goals,1,1
ft.shots,20,9
ft.shot_goal,1,1
ft.shot_on_target,4,3



--------------------------------------------------------------------------------



,0,1
game_id,1910843,1910843
game_date,2026-04-04 14:30:00,2026-04-04 14:30:00
week,28,28
team_name,Freiburg,Bayern Munich
game_venue,home,away
ft.possession,0.306186,0.693814
ft.goals,2,3
ft.shots,13,21
ft.shot_goal,2,3
ft.shot_on_target,4,9



--------------------------------------------------------------------------------



In [ ]:
# # season = config.get("season").get("year")
# db[mongo_config.get('collection').get("collection_processed_events")].delete_many({})
# db[mongo_config.get('collection').get("collection_team_game_stats")].delete_many({})
# db[mongo_config.get('collection').get("collection_player_game_stats")].delete_many({})

In [ ]:
# games = pd.DataFrame(db[mongo_config.get('collection').get("collection_raw_events")].find({'season': season}))

In [ ]:
# games.groupby('week')['game_id'].nunique()